## Imports

In [29]:
import pandas as pd
from pathlib import Path

## Paths for Project

In [30]:
ROOT_DIR = Path.cwd().parent

RAW_DIR = ROOT_DIR / "data" / "raw"
INTERIM_DIR = ROOT_DIR / "data" / "interim"

print(RAW_DIR)
print(INTERIM_DIR)

c:\Repos\capstone-hospital-quality\data\raw
c:\Repos\capstone-hospital-quality\data\interim


## Load Files with Reusable Loader

In [31]:
def load_cms_file(file_name: str) -> pd.DataFrame:
    """Load a CMS file and detect whether it uses tabs or commas."""
    file_path = RAW_DIR / file_name

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    with file_path.open("r", encoding="utf-8-sig") as file:
        header = file.readline()

    separator = "\t" if header.count("\t") > header.count(",") else ","

    df = pd.read_csv(
        file_path,
        sep=separator,
        dtype={"Facility ID": "string"},
        low_memory=False,
    )

    delimiter_name = "TAB" if separator == "\t" else "COMMA"
    print(f"{file_name}: {df.shape}, delimiter={delimiter_name}")

    return df

# Load Datasets

In [32]:
hospital_general = load_cms_file("hospital_general_info.csv")
hcahps = load_cms_file("HCAHPS.csv")
hospital_infections = load_cms_file("hospital_infections.csv")
hospital_mortality = load_cms_file("hospital_mortality.csv")
hospital_readmissions = load_cms_file("hospital_readmissions.csv")

hospital_general_info.csv: (5432, 38), delimiter=TAB
HCAHPS.csv: (325856, 22), delimiter=TAB
hospital_infections.csv: (172512, 15), delimiter=TAB
hospital_mortality.csv: (95840, 18), delimiter=TAB
hospital_readmissions.csv: (67088, 20), delimiter=TAB


# Confirm Shapes

In [33]:
print(hospital_general.shape)
print(hcahps.shape)
print(hospital_infections.shape)
print(hospital_mortality.shape)
print(hospital_readmissions.shape)

(5432, 38)
(325856, 22)
(172512, 15)
(95840, 18)
(67088, 20)


## Cleaning

In [34]:
def clean_column_names(df):
    """Standardize dataframe column names."""

    df = df.copy()

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace("__", "_", regex=False)
    )

    return df

In [35]:
hospital_general = clean_column_names(hospital_general)
hcahps = clean_column_names(hcahps)
hospital_infections = clean_column_names(hospital_infections)
hospital_mortality = clean_column_names(hospital_mortality)
hospital_readmissions = clean_column_names(hospital_readmissions)

## Replace Missing Values

In [36]:
def replace_not_available(df):

    df = df.replace(
        [
            "Not Available",
            "Not Applicable",
            "Not Available ",
            "Not Available.",
        ],
        pd.NA,
    )

    return df

In [37]:
hospital_general = replace_not_available(hospital_general)
hcahps = replace_not_available(hcahps)
hospital_infections = replace_not_available(hospital_infections)
hospital_mortality = replace_not_available(hospital_mortality)
hospital_readmissions = replace_not_available(hospital_readmissions)

## Recheck for Missing Values

In [38]:
hospital_general.isna().sum().sort_values(ascending=False)
hcahps.isna().sum().sort_values(ascending=False)
hospital_infections.isna().sum().sort_values(ascending=False)
hospital_mortality.isna().sum().sort_values(ascending=False)
hospital_readmissions.isna().sum().sort_values(ascending=False)

number_of_patients             58753
number_of_patients_returned    58753
footnote                       34496
denominator                    31857
score                          31857
higher_estimate                31857
lower_estimate                 31857
compared_to_national           15988
city_town                          0
address                            0
facility_name                      0
facility_id                        0
measure_id                         0
measure_name                       0
county_parish                      0
telephone_number                   0
zip_code                           0
state                              0
start_date                         0
end_date                           0
dtype: int64

# Verify duplicates

In [39]:
hospital_general.duplicated().sum()

np.int64(0)